## Ingest circuits.csv file


Read the file using spark dataframe reader API



Add Metadata Columns



Source File




Ingestion Timestamp






Write to bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/02_bronze_helpers"

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"

target_path = "s3://bizf1project123/Bronze/Kalashri_increment/circuits"


In [0]:
print("Batch ID:", v_batch_id)
print("Source file:", source_file)

## Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId',   StringType()),
    StructField("url",         StringType()),
    StructField("circuitName", StringType()),
    StructField("lat",         DoubleType()),
    StructField("long",        DoubleType()),
    StructField("locality",    StringType()),
    StructField("country",     StringType())
])

In [0]:
circuits_df = (
    spark.read
         .format('csv')
         .option('header', 'true')
#         .option('inferSchema', 'true')
         .option('mode', 'FAILFAST')
         .schema(circuits_schema)
         .load(source_file)
)

In [0]:
display(circuits_df)

## Step 2 - Add Metadata Columns


- Source File

- Ingestion Timestamp

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_final_df)

## Step 3 - Write to bronze delta table

In [0]:
write_to_bronze(
    input_df=circuits_final_df,
    target_path=target_path,
    batch_id=v_batch_id
)

In [0]:
%sql
SELECT *
FROM delta.`s3://bizf1project123/Bronze/Kalashri_increment/circuits`;

In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Bronze/Kalashri_increment/circuits",
#     True
# )

In [0]:
# display(dbutils.fs.ls(landing_folder_path))

In [0]:
# print("Batch ID =", v_batch_id)
# print("Landing folder =", landing_folder_path)
# print("Source file =", source_file)